In [0]:
# Cell 1: Widgets for interactivity
dbutils.widgets.dropdown("forecast_days", "7", ["7", "14", "30"], "Forecast Horizon (days)")
dbutils.widgets.text("city", "Chennai", "City")


In [0]:
# Cell 2: Read widget values and load latest forecast
n_days = int(dbutils.widgets.get("forecast_days"))
city   = dbutils.widgets.get("city")

df_forecast = spark.table("weather_forecast_gold").toPandas().sort_values("date")
df_actual   = spark.table("weather_silver").toPandas().sort_values("date")



In [0]:
# Cell 3: Dashboard plot — actual + forecast + confidence interval
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

fig, ax = plt.subplots(figsize=(13, 5))
last_60 = df_actual.tail(60)

ax.plot(pd.to_datetime(last_60["date"]), last_60["avg_temp"],
        label="Actual (last 60 days)", color="#1D9E75", linewidth=1.5)
ax.plot(pd.to_datetime(df_forecast["date"].head(n_days)), df_forecast["forecast_temp"].head(n_days),
        label=f"SARIMA Forecast ({n_days}d)", color="#D85A30", linewidth=2, linestyle="--")
ax.fill_between(
    pd.to_datetime(df_forecast["date"].head(n_days)),
    df_forecast["lower_ci"].head(n_days),
    df_forecast["upper_ci"].head(n_days),
    alpha=0.2, color="#D85A30", label="95% Confidence interval"
)
ax.axvline(pd.to_datetime("2024-01-01"), linestyle=":", color="gray", alpha=0.7)
ax.set_title(f"Temperature Forecast — {city}  |  Horizon: {n_days} days", fontsize=13)
ax.set_ylabel("Temperature (°C)")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
ax.legend(loc="upper left")
plt.tight_layout()
display(fig)



In [0]:
# Cell 4: KPI summary table
summary = {
    "Metric": ["Forecast start", "Forecast end", "Avg forecast temp", "Min forecast", "Max forecast"],
    "Value": [
        str(df_forecast["date"].min().date()),
        str(df_forecast["date"].head(n_days).max().date()),
        f"{df_forecast['forecast_temp'].head(n_days).mean():.1f} °C",
        f"{df_forecast['forecast_temp'].head(n_days).min():.1f} °C",
        f"{df_forecast['forecast_temp'].head(n_days).max():.1f} °C",
    ]
}
display(spark.createDataFrame(pd.DataFrame(summary)))



In [0]:
# Cell 5: Auto-refresh loop (simulates live dashboard)
import time
for i in range(5):
    print(f"[Refresh {i+1}/5] Latest forecast loaded at {pd.Timestamp.now().strftime('%H:%M:%S')}")
    # In real use: re-fetch API, run inference, update Delta table here
    time.sleep(3)